# Budowanie, trenowanie i testowanie modelu

In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# Dodanie dźwięku przy długich komórkach
import subprocess
def play_sound():
    sound_file = '/mnt/d/Backup/INZ/msg.ogg'
    subprocess.run(['ffplay', '-nodisp', '-autoexit', sound_file], capture_output=True)

Wczytanie odpowiednich danych

In [ ]:
SAVE_DIR = Path("/mnt/d/Backup/MAGISTERSKIE/outputs/MODEL")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(str(SAVE_DIR)+'/df_training.csv')
ds = df.to_numpy()
df.head()

# Wstępne Przetwarzanie Danych

Rozdzielenie danych na wejściowe i wyjściowe

In [ ]:
x_ds = ds[:, 0:17] # 17 danych - 3*3Δxyz + 6*dJ + 2*dG
y_ds = ds[:, 17:]  # 8  danych - 6*pred_dJ + 2*pred_dG
print(x_ds.shape)
print(y_ds.shape)
print(x_ds[0])
print(y_ds[0])

Rozdzielenie danych na treningowe i testowe

In [ ]:
rand = np.random.permutation(len(x_ds))
x_ds_shuffled = x_ds[rand]
y_ds_shuffled = y_ds[rand]

split_idx = int(len(x_ds_shuffled) * 0.9)
x_train = x_ds_shuffled[:split_idx]
x_test = x_ds_shuffled[split_idx:]
y_train = y_ds_shuffled[:split_idx]
y_test = y_ds_shuffled[split_idx:]

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

print(x_train[0])
print(x_test[0])

Normalizowanie danych

In [ ]:
#"""
x_train_mean = np.mean(x_train, axis=0)
x_train_std = np.std(x_train, axis=0)

x_train = (x_train - x_train_mean) / x_train_std
x_test = (x_test - x_train_mean) / x_train_std

y_train_mean = np.mean(y_train, axis=0)
y_train_variance = np.var(y_train, axis=0)

print(x_train[0])
print(x_test[0])
#""";

Sprawdzenie histogramów danych

In [ ]:
"""
for i in range(x_train.shape[1]):
  plt.figure(figsize=(10, 5))
  plt.hist(x_train[:, i], bins=100, alpha=0.5, label=str(df.columns[i] + ' trening'))
  plt.hist(x_test[:, i], bins=100, alpha=0.5, label=str(df.columns[i] + ' test'))
  plt.title(df.columns[i])
  # plt.xlabel(df.columns[i])
  # plt.ylabel('Frequency')
  plt.legend()
  plt.show()

for i in range(y_train.shape[1]):
  plt.figure(figsize=(10, 5))
  plt.hist(y_train[:, i], bins=100, alpha=0.5, label=str(df.columns[i+x_train.shape[1]] + ' trening'))
  plt.hist(y_test[:, i], bins=100, alpha=0.5, label=str(df.columns[i+x_train.shape[1]] + ' test'))
  plt.title(df.columns[i+x_train.shape[1]])
  # plt.xlabel(df.columns[i+x_train.shape[1]])
  # plt.ylabel('Frequency')
  plt.legend()
  plt.show()
#"""

In [ ]:
# Czytelniejsze nazwy do wykresów
position_labels = [
    'Pozycja X1 [m]',
    'Pozycja Y1 [m]',
    'Pozycja Z1 [m]',
    'Pozycja X2 [m]',
    'Pozycja Y2 [m]',
    'Pozycja Z2 [m]',
    'Pozycja X3 [m]',
    'Pozycja Y3 [m]',
    'Pozycja Z3 [m]',
]

joint_labels = [
    'Przegub 1 (J1)',
    'Przegub 2 (J2)',
    'Przegub 3 (J3)',
    'Przegub 4 (J4)',
    'Przegub 5 (J5)',
    'Przegub 6 (J6)',
    'Chwytak 1 (G1)',
    'Chwytak 2 (G2)',
]

delta_joint_labels = [
    'Zmiana przegubu 1 (dJ1)',
    'Zmiana przegubu 2 (dJ2)',
    'Zmiana przegubu 3 (dJ3)',
    'Zmiana przegubu 4 (dJ4)',
    'Zmiana przegubu 5 (dJ5)',
    'Zmiana przegubu 6 (dJ6)',
    'Zmiana chwytaka 1 (dG1)',
    'Zmiana chwytaka 2 (dG2)',
]

# 1. Histogramy przemieszczeń (3x3Δxyz -> 9 wykresów)
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Przemieszczenia (Δx, Δy, Δz)', fontsize=16)
for i, ax in enumerate(axes.flat):
    if i < 9:
        ax.hist(df.iloc[:, i], bins=100, color='skyblue', alpha=0.7)
        ax.set_title(f"Δ{df.columns[i]}")
        ax.set_xlabel('Różnica współrzędnych [px]')
        ax.set_ylabel('Liczba próbek')
plt.tight_layout()
plt.show()

# 2. Histogramy pozycji (J1-6 + G1-2 -> 8 wykresów)
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Pozycje przegubów i chwytaka', fontsize=16)
for i, ax in enumerate(axes.flat):
    if i < 8:
        ax.hist(df.iloc[:, 9 + i], bins=100, color='lightgreen', alpha=0.7)
        ax.set_title(joint_labels[i])
        ax.set_xlabel('Kąt [rad]')
        ax.set_ylabel('Liczba próbek')
    else:
        ax.axis('off')  # Ukryj ostatni, pusty wykres
plt.tight_layout()
plt.show()

# 3. Histogramy różnic (dJ1-6 + dG1-2 -> 8 wykresów)
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Zmiany przegubów i chwytaka', fontsize=16)
for i, ax in enumerate(axes.flat):
    if i < 8:
        ax.hist(df.iloc[:, 17 + i], bins=100, color='salmon', alpha=0.7)
        ax.set_title(delta_joint_labels[i])
        ax.set_xlabel('Zmiana kąta [rad]')
        ax.set_ylabel('Liczba próbek')
    else:
        ax.axis('off')  # Ukryj ostatni, pusty wykres
plt.tight_layout()
plt.show()

# Tworzenie i Konfiguracja Modelu

Podstawowe parametry modelu

In [ ]:
OUTPUTS_PATH = Path("/mnt/d/Backup/MAGISTERSKIE/outputs/MODEL/model_outputs")
MODEL_PATH = Path("/mnt/d/Backup/MAGISTERSKIE/outputs/MODEL/model")

OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)
MODEL_PATH.mkdir(parents=True, exist_ok=True)

input_shape = x_train.shape[-1]
output_shape = y_train.shape[-1]
print(f"input_shape: {input_shape}, output_shape: {output_shape}")

Lista callback do uczenia

In [ ]:
callbacks_list = [
  tf.keras.callbacks.ModelCheckpoint(
    filepath=str(MODEL_PATH)+'/saved_model.keras',
    monitor='val_loss',
    save_best_only=True
  ),
  tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=6,
    verbose=True
  ),
  tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
  )]

## Wariant 1 - Budowanie modelu ręcznie

Parametry modelu

In [ ]:
neurons = 320
dropout = 0.1

In [ ]:
"""
inputs = tf.keras.Input(shape=(input_shape,))

# augmentacja
#x = tf.keras.layers.GaussianNoise(0.1)(inputs)

x = tf.keras.layers.Dense(neurons, activation='relu', activity_regularizer=tf.keras.regularizers.l2(1e-4))(inputs)
x = tf.keras.layers.BatchNormalization()(x)
#x = tf.keras.layers.Dropout(dropout/2)(x)

# Pierwszy blok residual
x_skip = x
x = tf.keras.layers.Dense(neurons*2, activation='relu', activity_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = tf.keras.layers.BatchNormalization()(x)
#x = tf.keras.layers.Dropout(dropout)(x)
x = tf.keras.layers.Dense(neurons, activation='relu', activity_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Add()([x, x_skip])

# Drugi blok residual
x_skip = x
x = tf.keras.layers.Dense(neurons*2, activation='relu', activity_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = tf.keras.layers.BatchNormalization()(x)
#x = tf.keras.layers.Dropout(dropout)(x)
x = tf.keras.layers.Dense(neurons, activation='relu', activity_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Add()([x, x_skip])

outputs_normalized = tf.keras.layers.Dense(output_shape, name='normalized_dense')(x)

outputs = tf.keras.layers.Normalization(
    mean=y_train_mean, 
    variance=y_train_variance, 
    invert=True, 
    name='denormalization'
)(outputs_normalized)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.summary()
#""";

## Wariant 2 - Budowanie modelu za pomocą tunera

In [ ]:
"""
import keras_tuner as kt

input_shape = x_train.shape[-1]
output_shape = y_train.shape[-1]

def build_model(hp):
    hp: kt.HyperParameters
    
    num_blocks = hp.Int('num_blocks', min_value=4, max_value=8, step=1)
    neurons = hp.Int('neurons', min_value=64, max_value=640, step=64)
    learning_rate = hp.Float('learning_rate', min_value=1e-5, max_value=1e-2, sampling='log')
    
    inputs = tf.keras.Input(shape=(input_shape,))
    x = inputs
    x = tf.keras.layers.Dense(neurons, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    
    for i in range(num_blocks):
        x_skip = x
        x = tf.keras.layers.Dense(neurons*2, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.1)(x)
        x = tf.keras.layers.Dense(neurons, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Add()([x, x_skip])
    
    outputs_normalized = tf.keras.layers.Dense(output_shape, name='normalized_dense')(x)
    outputs = tf.keras.layers.Normalization(mean=y_train_mean, variance=y_train_variance, invert=True, name='denormalization')(outputs_normalized)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )
    return model

tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=16,
    executions_per_trial=1,
    #directory='ktuner_dir',
    project_name='diff_pred_tuning'
)

tuner.search(x_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=256,
    callbacks=callbacks_list)
best_hp = tuner.get_best_hyperparameters(1)[0]
print("Best number of blocks:", best_hp.get('num_blocks'))
print("Best neurons:", best_hp.get('neurons'))
print("Best learning rate:", best_hp.get('learning_rate'))

play_sound()
#"""

Tworzenie szkieletu modelu i zapisanie do pliku

In [ ]:
# model = build_model(best_hp)
# model.save(str(MODEL_PATH)+"/new_model_tuned.keras")

# Proces Uczenia

Odczytanie znalezionego szkieletu modelu

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH / "new_model_tuned.keras")

In [ ]:
"""
base_learning_rate = 1e-2
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    base_learning_rate,
    decay_steps=2318,
    decay_rate=0.8,
    staircase=True)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
              loss= tf.keras.losses.MeanSquaredError(),
              metrics = [tf.keras.metrics.RootMeanSquaredError()])
#""";

Proces uczenia

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=300,
    batch_size=256,
    callbacks=callbacks_list
)
play_sound()

In [ ]:
model.evaluate(x=x_test, y=y_test, batch_size=256, verbose=1)

## Testowanie uzyskanego modelu

Zapisanie wyników predykcji

In [ ]:
model = tf.keras.models.load_model(str(MODEL_PATH)+'/saved_model.keras')

In [ ]:
result = model.predict(x=x_test)

Wyświetlenie błędów na poszczególnych przegubach

In [ ]:
diff_test = np.rad2deg(np.mean(np.abs(result-y_test), axis=0))

joints_error_dict = dict(zip(df.columns[17:], diff_test))
for joint, error in joints_error_dict.items():
    print(f"{joint}: {error:.2f} stopni")

Wyświetlenie średniej i mediane błędu absolutnego

In [ ]:
mean_test = np.rad2deg(np.mean(np.abs(result-y_test)))
median_test = np.rad2deg(np.median(np.abs(result-y_test)))

print(f"Średnia: {mean_test:.2f} stopni")
print(f"Mediana: {median_test:.2f} stopni")

## Zapisanie wyników do pliku .csv

In [ ]:
cols = [ 'J1', 'J2', 'J3', 'J4', 'J5', 'J6', 'G1', 'G2',
        'dJ1', 'dJ2', 'dJ3', 'dJ4', 'dJ5', 'dJ6', 'dG1', 'dG2',
        'pred_dJ1', 'pred_dJ2', 'pred_dJ3', 'pred_dJ4', 'pred_dJ5', 'pred_dJ6', 'pred_dG1', 'pred_dG2']

In [ ]:
print(result.shape)
print(x_test[:,9:].shape)
print(y_test.shape)

Zapisanie wyników do testowania na robocie

In [ ]:
wyniki = pd.concat([pd.DataFrame(x_test[:,9:]*x_train_std[9:]+x_train_mean[9:], columns=cols[:8]),  # Pozycja początkowa
                    pd.DataFrame(y_test, columns=cols[8:16]),                                       # Różnica faktyczna
                    pd.DataFrame(result, columns=cols[16:])], axis=1)                               # Różnica przewidywana
wyniki.to_csv(str(OUTPUTS_PATH)+"/wyniki.csv", index=False, header=True)

In [ ]:
wyniki.head(5)

In [ ]:
print("Mniejszy od 1 stopnia:")
for col in cols[8:16]:
    print(f"{col}: {np.sum(np.abs(np.rad2deg(wyniki[col])-np.rad2deg(wyniki['pred_'+col]))<1)} / {len(wyniki)} {np.sum(np.abs(np.rad2deg(wyniki[col])-np.rad2deg(wyniki['pred_'+col]))<1)/len(wyniki)*100:.2f}%")

In [ ]:
do_pracy = pd.DataFrame({'nazwa': cols[8:16],
                         'średnia': [(np.rad2deg(np.mean(np.abs(wyniki[col]-wyniki['pred_'+col])))).round(2) for col in cols[8:16]],
                         'mediana': [(np.rad2deg(np.median(np.abs(wyniki[col]-wyniki['pred_'+col])))).round(2) for col in cols[8:16]],
                         'odchylenie_standardowe': [(np.rad2deg(np.std(np.abs(wyniki[col]-wyniki['pred_'+col])))).round(2) for col in cols[8:16]],
                         'mniejsze_od_1': [np.sum(np.abs(np.rad2deg(wyniki[col])-np.rad2deg(wyniki['pred_'+col]))<1) for col in cols[8:16]],
                         'procent_mniejsze_od_1 [%]': [(np.sum(np.abs(np.rad2deg(wyniki[col])-np.rad2deg(wyniki['pred_'+col]))<1)/len(wyniki)*100).round(2) for col in cols[8:16]]}
                        )
do_pracy.head(8)
# skopiowanie ostatniego wiesza do schowka transponowana
do_pracy.iloc[-1].T.to_clipboard()
#do_pracy.iloc[-1].to_clipboard()

In [ ]:
RESULT_PATH = Path("/mnt/c/Users/rados/downloads")
df_wyniki = pd.read_csv(str(RESULT_PATH)+'/wyniki_vert_converted.csv')

# Obliczenie błędów i konwersja jednostek na metry (dzielenie przez 1000 zakłada, że dane bazowe są w mm)
df_wyniki['error_x_m'] = (df_wyniki['x_to_predicted'] - df_wyniki['x_to_ground_truth'])
df_wyniki['error_y_m'] = (df_wyniki['y_to_predicted'] - df_wyniki['y_to_ground_truth'])
df_wyniki['error_z_m'] = (df_wyniki['z_to_predicted'] - df_wyniki['z_to_ground_truth'])

# Obliczenie całkowitego błędu położenia w metrach
df_wyniki['error_magnitude_m'] = np.sqrt(df_wyniki[['error_x_m', 'error_y_m', 'error_z_m']].pow(2).sum(axis=1))

# Wyświetlenie histogramu błędu położenia w metrach
plt.figure(figsize=(10, 6))
plt.hist(df_wyniki['error_magnitude_m'], bins=30, color='skyblue', edgecolor='black')
plt.title('Histogram błędu położenia')
plt.xlabel('Błąd pozycji XYZ [m]')
plt.ylabel('Liczba próbek')
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

Histogram otrzymanych wyników

In [ ]:
RESULT_PATH = Path("/mnt/c/Users/rados/downloads")

# Załdowanie danych i obliczenie błędów
df_wyniki = pd.read_csv(str(RESULT_PATH)+'/wyniki_vert_converted.csv')
df_wyniki['error_x'] = df_wyniki['x_to_predicted'] - df_wyniki['x_to_ground_truth']
df_wyniki['error_y'] = df_wyniki['y_to_predicted'] - df_wyniki['y_to_ground_truth']
df_wyniki['error_z'] = df_wyniki['z_to_predicted'] - df_wyniki['z_to_ground_truth']
df_wyniki['error_magnitude'] = np.sqrt(df_wyniki[['error_x', 'error_y', 'error_z']].pow(2).sum(axis=1))
mean_error = df_wyniki['error_magnitude'].mean()
median_error = df_wyniki['error_magnitude'].median()

# Wyświetlenie histogramu błędu położenia
plt.figure()
plt.hist(df_wyniki['error_magnitude'], bins=30)
plt.axvline(mean_error, linestyle='--', linewidth=2, label=f'Średnia: {mean_error:.4f}', color='red')
plt.axvline(median_error, linestyle='--', linewidth=2, label=f'Mediana: {median_error:.4f}', color='orange')
plt.title('Histogram błędu położenia')
plt.xlabel('Błąd pozycji XYZ [m]')
plt.ylabel('Liczba próbek')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
print(df.columns)